# Лабораторная 2. Формирование отчётов в Apache Spark

**Задание:** Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

**Данные:**
- `posts_sample.xml` — выборка постов Stack Overflow
- `programming-languages.csv` — список языков программирования

**Результат:** набор таблиц «топ-10» для каждого года, сохранённый в формате Apache Parquet.

## 0. Установка PySpark и загрузка данных

In [5]:
!pip install pyspark -q

In [6]:
import os

for f in ["/content/posts_sample.xml", "/content/programming-languages.csv"]:
    print(f"{'✓' if os.path.exists(f) else '✗'} {f}")

✓ /content/posts_sample.xml
✓ /content/programming-languages.csv


## 1. Инициализация SparkSession

In [7]:
import re
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Lab2_TopLanguagesReport") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark version:", spark.version)

Spark version: 4.0.2


## 2. Загрузка списка языков программирования

In [8]:
languages_df = spark.read.csv("/content/programming-languages.csv", header=True)

languages_set = set(
    row["name"].strip().lower()
    for row in languages_df.collect()
    if row["name"] is not None
)

languages_broadcast = sc.broadcast(languages_set)
print(f"Загружено {len(languages_set)} языков")

Загружено 698 языков


## 3. Парсинг posts_sample.xml

In [9]:
raw_rdd = sc.textFile("/content/posts_sample.xml")
rows_rdd = raw_rdd.filter(lambda line: line.strip().startswith("<row"))
print(f"Всего строк с данными: {rows_rdd.count()}")

Всего строк с данными: 9883


In [10]:
def parse_row(line):
    results = []
    date_match = re.search(r'CreationDate="([^"]+)"', line)
    if not date_match:
        return results
    year = int(date_match.group(1)[:4])
    if year < 2010 or year > 2020:
        return results

    tags_match = re.search(r'Tags="([^"]*)"', line)
    if not tags_match:
        return results

    tags_decoded = tags_match.group(1).replace("&lt;", "<").replace("&gt;", ">")
    tags = re.findall(r"<([^>]+)>", tags_decoded)

    langs = languages_broadcast.value
    for tag in tags:
        if tag.lower() in langs:
            results.append(Row(year=year, language=tag.lower()))
    return results

parsed_rdd = rows_rdd.flatMap(parse_row)
print(f"Всего пар (год, язык): {parsed_rdd.count()}")

Всего пар (год, язык): 1699


## 4. Агрегация и построение топ-10 по годам

In [11]:
posts_df = spark.createDataFrame(parsed_rdd)

lang_counts = posts_df.groupBy("year", "language") \
    .agg(count("*").alias("count")) \
    .orderBy("year", col("count").desc())

window_spec = Window.partitionBy("year").orderBy(col("count").desc())

top10 = lang_counts \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .orderBy("year", "rank")

top10.show(110, truncate=False)

+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|javascript |20   |1   |
|2010|php        |15   |2   |
|2010|java       |15   |3   |
|2010|c          |8    |4   |
|2010|objective-c|7    |5   |
|2010|python     |6    |6   |
|2010|ruby       |6    |7   |
|2010|bash       |2    |8   |
|2010|delphi     |2    |9   |
|2010|perl       |2    |10  |
|2011|java       |26   |1   |
|2011|php        |20   |2   |
|2011|javascript |14   |3   |
|2011|c          |9    |4   |
|2011|objective-c|6    |5   |
|2011|python     |6    |6   |
|2011|ruby       |5    |7   |
|2011|perl       |3    |8   |
|2011|coldfusion |3    |9   |
|2011|r          |2    |10  |
|2012|java       |35   |1   |
|2012|php        |33   |2   |
|2012|javascript |32   |3   |
|2012|python     |18   |4   |
|2012|objective-c|11   |5   |
|2012|ruby       |9    |6   |
|2012|c          |9    |7   |
|2012|curl       |3    |8   |
|2012|haskell    |2    |9   |
|2012|xpath      |2    |10  |
|2013|java

## 5. Сохранение в Parquet

In [12]:
top10.write.mode("overwrite").parquet("top_languages_by_year.parquet")
print("Сохранено в top_languages_by_year.parquet")

Сохранено в top_languages_by_year.parquet


## 6. Проверка

In [13]:
result = spark.read.parquet("top_languages_by_year.parquet")
result.orderBy("year", "rank").show(110, truncate=False)

+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|javascript |20   |1   |
|2010|php        |15   |2   |
|2010|java       |15   |3   |
|2010|c          |8    |4   |
|2010|objective-c|7    |5   |
|2010|python     |6    |6   |
|2010|ruby       |6    |7   |
|2010|bash       |2    |8   |
|2010|delphi     |2    |9   |
|2010|perl       |2    |10  |
|2011|java       |26   |1   |
|2011|php        |20   |2   |
|2011|javascript |14   |3   |
|2011|c          |9    |4   |
|2011|objective-c|6    |5   |
|2011|python     |6    |6   |
|2011|ruby       |5    |7   |
|2011|perl       |3    |8   |
|2011|coldfusion |3    |9   |
|2011|r          |2    |10  |
|2012|java       |35   |1   |
|2012|php        |33   |2   |
|2012|javascript |32   |3   |
|2012|python     |18   |4   |
|2012|objective-c|11   |5   |
|2012|ruby       |9    |6   |
|2012|c          |9    |7   |
|2012|curl       |3    |8   |
|2012|haskell    |2    |9   |
|2012|xpath      |2    |10  |
|2013|java

In [14]:
spark.stop()